In [1]:
import sys
import subprocess
import pkgutil

required = [
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("datasets", "datasets"),
    ("transformers", "transformers"),
    ("snorkel", "snorkel"),
    ("wandb", "wandb"),
    ("tqdm", "tqdm"),
    ("reportlab", "reportlab"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("scikit-learn", "sklearn"),
]

missing = []
for pkg_name, import_name in required:
    if pkgutil.find_loader(import_name) is None:
        missing.append(pkg_name)

if missing:
    print("Missing packages detected:", missing)
    print("Installing missing packages.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages appear installed.")


All required packages appear installed.


C:\Users\govar\AppData\Local\Temp\ipykernel_8980\1569718781.py:21: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(import_name) is None:


In [2]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def top1_accuracy_from_logits(logits, targets):
    """Return top-1 accuracy percentage (0-100). logits: tensor (N,C), targets: tensor (N,)"""
    pred = logits.argmax(dim=1)
    correct = (pred == targets).float().sum().item()
    return 100.0 * correct / targets.size(0)

set_seed(42)

In [3]:
# CoNLL-2003 dataset load and W&B logging
from datasets import load_dataset
from collections import Counter
import wandb

PROJECT_NER = "Q1-weak-supervision-ner"

# Helper to compute token-level entity counts and sentence count
def compute_conll_stats(split_dataset):
    label_names = split_dataset.features["ner_tags"].feature.names
    n_sentences = len(split_dataset)
    n_tokens = 0
    entity_counts = Counter()
    for item in split_dataset:
        tokens = item["tokens"]
        tags = item["ner_tags"]
        n_tokens += len(tokens)
        for t in tags:
            label = label_names[t]
            if label == "O":
                continue
            if "-" in label:
                _, ent = label.split("-", 1)
            else:
                ent = label
            entity_counts[ent] += 1
    return {
        "n_sentences": n_sentences,
        "n_tokens": n_tokens,
        "entity_counts": dict(entity_counts),
        "label_names": label_names
    }
# Load dataset (this will download if not cached)
print("Loading eriktks/conll2003 from HuggingFace...")
ds = load_dataset("eriktks/conll2003")
print("Dataset loaded. Splits:", ds.keys())

# Compute stats for train split
train_split = ds["train"]
stats = compute_conll_stats(train_split)
print("Train stats:", stats)

# Log to W&B
run = wandb.init(project=PROJECT_NER, name="conll_stats_jupyter", reinit=True)
wandb.run.summary["num_sentences"] = stats["n_sentences"]
wandb.run.summary["num_tokens"] = stats["n_tokens"]
for ent, cnt in stats["entity_counts"].items():
    wandb.run.summary[f"entity_count_{ent}"] = cnt
wandb.config.update({"label_names": stats["label_names"]})
print("Logged dataset summary to W&B in project", PROJECT_NER)
wandb.finish()

Loading eriktks/conll2003 from HuggingFace...


Using the latest cached version of the dataset since eriktks/conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\govar\.cache\huggingface\datasets\eriktks___conll2003\default\0.0.0\ce85b39f9dd99f552d0739d456814e95fb6a39b0 (last modified on Mon Oct 13 19:51:18 2025).


Dataset loaded. Splits: dict_keys(['train', 'validation', 'test'])
Train stats: {'n_sentences': 14041, 'n_tokens': 203621, 'entity_counts': {'ORG': 10025, 'MISC': 4593, 'PER': 11128, 'LOC': 8297}, 'label_names': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']}


wandb: Currently logged in as: 142502032 (ir2023) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Logged dataset summary to W&B in project Q1-weak-supervision-ner


entity_count_LOC,8297
entity_count_MISC,4593
entity_count_ORG,10025
entity_count_PER,11128
num_sentences,14041
num_tokens,203621


In [4]:
# Snorkel labeling functions: prepare token-level dataframe, define LFs, compute L matrix
import pandas as pd
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
try:
    from snorkel.labeling.model import ABSTAIN
except Exception:
    ABSTAIN = -1


LABEL_MAP = {"PER": 0, "LOC": 1, "ORG": 2, "MISC": 3}
INV_LABEL_MAP = {v:k for k,v in LABEL_MAP.items()}

# Prepare a token-level dataframe from conll train split
rows = []
label_names = train_split.features["ner_tags"].feature.names
for i, item in enumerate(train_split):
    tokens = item["tokens"]
    tags = item["ner_tags"]
    for tok, tg in zip(tokens, tags):
        label_name = label_names[tg]
        if label_name == "O":
            true_label = -1
        else:
            if "-" in label_name:
                _, ent = label_name.split("-", 1)
            else:
                ent = label_name
            true_label = LABEL_MAP.get(ent, -1)
        rows.append({"sentence_id": i, "token": tok, "true_label": int(true_label)})

df_tokens = pd.DataFrame(rows)
print("Token-level dataframe shape:", df_tokens.shape)

# Define labeling functions
import re

@labeling_function()
def lf_year_as_misc(x):
    tok = str(x.token)
    if re.fullmatch(r"(19|20)\d{2}", tok):
        return LABEL_MAP["MISC"]
    return ABSTAIN

@labeling_function()
def lf_org_suffix(x):
    tok = str(x.token)
    t = tok.lower()
    # check simple suffixes; includes forms with/without period
    if t.endswith("inc.") or t.endswith("inc") or t.endswith("corp.") or t.endswith("corp") or t.endswith("ltd.") or t.endswith("ltd"):
        return LABEL_MAP["ORG"]
    return ABSTAIN

lfs = [lf_year_as_misc, lf_org_suffix]
applier = PandasLFApplier(lfs=lfs)
L = applier.apply(df_tokens)
print("L matrix shape:", L.shape)

# Evaluate coverage & LF-level accuracy using available gold labels (ignore true_label == -1)
import numpy as np

y_true = df_tokens["true_label"].to_numpy()

lf_results = []
for j, lf in enumerate(lfs):
    col = L[:, j]
    non_abstain = col != ABSTAIN
    coverage = non_abstain.mean()
    valid_mask = non_abstain & (y_true != -1)
    accuracy = float(np.nan)
    if valid_mask.sum() > 0:
        accuracy = (col[valid_mask] == y_true[valid_mask]).mean()
    lf_results.append({"lf_index": j, "coverage": float(coverage), "accuracy": float(accuracy)})
    print(f"LF {j} coverage={coverage:.6f} accuracy={accuracy}")

# Log LF metrics to W&B
run = wandb.init(project=PROJECT_NER, name="snorkel_lfs_jupyter", reinit=True)
for r in lf_results:
    wandb.log({f"LF_{r['lf_index']}_coverage": r['coverage'], f"LF_{r['lf_index']}_accuracy": r['accuracy']})
# Save LF analysis CSV
try:
    analysis = LFAnalysis(L=L, lfs=lfs).lf_summary()
    df_analysis = pd.DataFrame(analysis)
    df_analysis.to_csv("lf_analysis.csv", index=False)
    wandb.save("lf_analysis.csv")
    print("Saved lf_analysis.csv and uploaded to W&B")
except Exception as e:
    print("LFAnalysis failed:", e)
wandb.finish()

Token-level dataframe shape: (203621, 3)


100%|███████████████████████████████████████████████████████████████████████| 203621/203621 [00:03<00:00, 52238.46it/s]


L matrix shape: (203621, 2)
LF 0 coverage=0.002667 accuracy=0.75
LF 1 coverage=0.000840 accuracy=1.0


LFAnalysis failed: [WinError 1314] A required privilege is not held by the client: 'C:\\Users\\govar\\Documents\\MLOPS\\Assignments\\Assignment_5\\lf_analysis.csv' -> 'C:\\Users\\govar\\Documents\\MLOPS\\Assignments\\Assignment_5\\wandb\\run-20251013_222805-4bhcblw4\\files\\lf_analysis.csv'


LF_0_accuracy,▁
LF_0_coverage,▁
LF_1_accuracy,▁
LF_1_coverage,▁
LF_0_accuracy,0.75
LF_0_coverage,0.00267
LF_1_accuracy,1
LF_1_coverage,0.00084


In [6]:
# MajorityLabelVoter aggregation and evaluation
from snorkel.labeling.model.baselines import MajorityLabelVoter

maj = MajorityLabelVoter(cardinality=4)
y_maj = maj.predict(L=L)

mask = (y_maj != -1) & (y_true != -1)
maj_coverage = (y_maj != -1).mean()
maj_accuracy = float(np.nan)
if mask.sum() > 0:
    maj_accuracy = (y_maj[mask] == y_true[mask]).mean()

print(f"MajorityVoter coverage={maj_coverage:.6f} accuracy={maj_accuracy}")

run = wandb.init(project=PROJECT_NER, name="majority_voter_jupyter", reinit=True)
wandb.log({"majority_coverage": float(maj_coverage), "majority_accuracy": float(maj_accuracy)})
# Save L and y arrays for inspection
np.save("L_matrix.npy", L)
np.save("y_true.npy", y_true)
artifact_L = wandb.Artifact("L_matrix", type="dataset")
artifact_L.add_file("L_matrix.npy")
wandb.log_artifact(artifact_L)
artifact_y = wandb.Artifact("y_true", type="dataset")
artifact_y.add_file("y_true.npy")
wandb.log_artifact(artifact_y)
wandb.finish()


MajorityVoter coverage=0.003507 accuracy=0.9940828402366864


majority_accuracy,▁
majority_coverage,▁
majority_accuracy,0.99408
majority_coverage,0.00351


majority_accuracy,▁
majority_coverage,▁
majority_accuracy,0.99408
majority_coverage,0.00351


In [ ]:
# CIFAR sequential training (ResNet18) — small helper experiment
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

PROJECT_CIFAR = "cifar-sequential"

# Dataloaders
def get_dataloaders(name, batch_size=128, num_workers=4):
    if name == "CIFAR10":
        mean = [0.4914, 0.4822, 0.4465]
        std = [0.2470, 0.2435, 0.2616]
    else:
        mean = [0.5071, 0.4867, 0.4408]
        std = [0.2675, 0.2565, 0.2761]

    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    if name == "CIFAR10":
        trainset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
        testset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)
        num_classes = 10
    else:
        trainset = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_train)
        testset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_test)
        num_classes = 100

    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return trainloader, testloader, num_classes

# Build adapted ResNet18 for CIFAR

def build_resnet18(num_classes, device):
    model = models.resnet18(pretrained=False)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

# Train & eval

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += X.size(0)
    return running_loss / total, 100.0 * correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            running_loss += loss.item() * X.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += X.size(0)
    return running_loss / total, 100.0 * correct / total

# Experiment runner

def run_sequential_experiment(first, second, epochs=10, batch_size=128, device=None, run_name_suffix="run"):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    run = wandb.init(project=PROJECT_CIFAR, name=f"{first}_then_{second}_{run_name_suffix}", reinit=True)

    # Phase 1
    train1, test1, num_cls1 = get_dataloaders(first, batch_size=batch_size)
    model = build_resnet18(num_cls1, device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[int(epochs*0.6), int(epochs*0.8)], gamma=0.1)
    print(f"Training phase 1: {first} for {epochs} epochs on {device}")
    for ep in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train1, optimizer, criterion, device)
        val_loss, val_acc = eval_epoch(model, test1, criterion, device)
        scheduler.step()
        wandb.log({"phase": first, "epoch": ep, "train_loss": tr_loss, "train_acc": tr_acc, "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{first}] epoch {ep}: train_acc={tr_acc:.2f} val_acc={val_acc:.2f}")

    # Save checkpoint
    ckpt_dir = "checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt1 = f"{ckpt_dir}/{first}_model.pth"
    torch.save(model.state_dict(), ckpt1)
    wandb.save(ckpt1)

    # Phase 2: adjust final fc if class count differs
    train2, test2, num_cls2 = get_dataloaders(second, batch_size=batch_size)
    if num_cls2 != num_cls1:
        model.fc = nn.Linear(model.fc.in_features, num_cls2).to(device)
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
        scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[int(epochs*0.6), int(epochs*0.8)], gamma=0.1)

    print(f"Training phase 2: {second} for {epochs} epochs on {device}")
    for ep in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train2, optimizer, criterion, device)
        val_loss, val_acc = eval_epoch(model, test2, criterion, device)
        scheduler.step()
        wandb.log({"phase": second, "epoch": epochs + ep, "train_loss": tr_loss, "train_acc": tr_acc, "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{second}] epoch {ep}: train_acc={tr_acc:.2f} val_acc={val_acc:.2f}")

    ckpt2 = f"{ckpt_dir}/{second}_after_{first}_model.pth"
    torch.save(model.state_dict(), ckpt2)
    wandb.save(ckpt2)
    wandb.finish()
    
# Experiment A: CIFAR-100 then CIFAR-10 (100 epochs each — heavy)
run_sequential_experiment(
    first="CIFAR100",
    second="CIFAR10",
    epochs=100,
    batch_size=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    run_name_suffix="expA"
)

# Experiment B: CIFAR-10 then CIFAR-100 (100 epochs each)
run_sequential_experiment(
    first="CIFAR10",
    second="CIFAR100",
    epochs=100,
    batch_size=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    run_name_suffix="expB"
)

100%|███████████████████████████████████████████████████████████████████████████████| 169M/169M [42:40<00:00, 66.0kB/s]
C:\Users\govar\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\govar\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Training phase 1: CIFAR100 for 100 epochs on cpu
